In [1]:
import json
import os
from itertools import islice
from pathlib import Path
from pickle import dump, load
from time import time

import numpy as np
import pyspark
import pyspark.sql.functions as sf
import pyarrow as pa
from pyspark.sql.types import (
    ArrayType,
    FloatType,
    IntegerType,
    StringType,
    StructField,
    StructType,
)
from tqdm import tqdm
from vastdb.session import Session

with open("/scratch/gw2338/vast/data-lake-main/spark/scripts/.env") as f:
    envvars = dict(x.strip().split("=") for x in f.readlines())

In [2]:
from colabfit.tools.vast.configuration import AtomicConfiguration
from colabfit.tools.vast.configuration_set import ConfigurationSet
from colabfit.tools.vast.database import DataManager, VastDataLoader, batched
from colabfit.tools.vast.property import PropertyMap, property_info
from colabfit.tools.property_definitions import (
    atomic_forces_pd,
    cauchy_stress_pd,
    energy_pd,
)
from colabfit.tools.vast.schema import *
from colabfit.tools.vast.utilities import unstring_df_val, stringify_df_val_udf, str_to_arrayof_str, str_to_arrayof_int, spark_schema_to_arrow_schema
from pyspark.sql import SparkSession, Row

In [3]:
jars = os.getenv("VASTDB_CONNECTOR_JARS")
spark = (
    SparkSession.builder.appName("colabfit").config("spark.jars", jars).config('spark.sql.shuffle.partitions', 2).getOrCreate())

loader = VastDataLoader(
    table_prefix="ndb.colabfit.dev",
)
loader.set_spark_session(spark)
access_key = envvars.get("SPARK_ID")
access_secret = envvars.get("SPARK_KEY")
endpoint = envvars.get("SPARK_ENDPOINT")

25/03/10 10:41:13 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable
Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).
25/03/10 10:41:17 WARN SparkSession: Using an existing Spark session; only runtime SQL configurations will take effect.


In [4]:
sess = Session(access=access_key, secret=access_secret, endpoint=endpoint)

In [5]:
fps = sorted(
    [
        str(x) for x in Path(
            "/scratch/gw2338/vast/data-lake-main/spark/scripts/gw_scripts/fix_duplicate_co_ids/co_duplicate_parquets/"
        ).glob('*.parquet')]
)

In [6]:
df = spark.read.parquet(*fps)
df.select('id').distinct().count()

1182

In [7]:
df.select('$row_id').write.mode('overwrite').parquet('/scratch/gw2338/vast/data-lake-main/spark/scripts/gw_scripts/fix_duplicate_co_ids/row_ids_to_drop.parquet')
df.count()

2376

In [8]:
update_cols = ['labels','names','dataset_ids']

In [9]:
from itertools import chain
from ast import literal_eval

In [10]:
set(list(chain.from_iterable([literal_eval(x['dataset_ids']) for x in df.select('dataset_ids').distinct().collect()])))

{'DS_4vdrw3cfi4s7_0',
 'DS_abagltajle7q_0',
 'DS_c4s38mdirjf7_0',
 'DS_gxjhn6vdjnxg_0',
 'DS_h7gnyidyqcxe_0',
 'DS_krd83tt1g6wd_0',
 'DS_mm4npn96qxo1_0',
 'DS_qrkr2xiw1wtp_0',
 'DS_s6gf4z2hcjqy_0',
 'DS_sng40qq19dak_0',
 'DS_vxd0mrt5it19_0',
 'DS_zduxfk2oohzc_0'}

In [11]:
df2 = df.select([col if col not in ['names','dataset_ids','labels'] else str_to_arrayof_str(col).alias(col) for col in df.columns])

In [12]:
@sf.udf('array<string>')
def concat_cols(col1):
    if not isinstance(col1[0], list):
        return col1
    x = []
    return list(set([x.extend(y) for y in col1]))

In [13]:
def deduplicate_co_df(co_df):
    """Combine values across duplicate CO rows."""
    grouped_id = co_df.groupBy("id")
    merged_names = grouped_id.agg(sf.flatten(
        sf.array_distinct(sf.collect_list("names"))).alias("names")
    )
    co_df = co_df.dropDuplicates(["id"])
    co_df = co_df.drop("names").join(merged_names, on="id", how="inner")
    if co_df.select("labels").filter(sf.col("labels").isNotNull()).count() > 0:
        merged_labels = grouped_id.agg(
            sf.flatten(sf.array_distinct(sf.collect_list("labels"))).alias("labels")
        )
        co_df = co_df.drop("labels").join(merged_labels, on="id", how="inner")
    merged_ds_ids = grouped_id.agg(sf.flatten(
        sf.array_distinct(sf.collect_list("dataset_ids"))).alias("dataset_ids")
    )
    co_df = co_df.drop("dataset_ids").join(merged_ds_ids, on="id", how="inner")
    
    co_df = co_df.select(config_schema.fieldNames())
    return co_df

In [14]:
df3 = deduplicate_co_df(df2)

In [15]:
df3.count()

1182

In [16]:
df4 = df3.select([col if col not in update_cols else stringify_df_val_udf(col).alias(col) for col in config_schema.fieldNames()])

In [17]:
df4.first()

Row(id='CO_1000004848526301716883101', hash='10000048485263017168831015908510373713380818269381667947754271542883863097145146009717605405373851448841646959686261323792667237781532903767731325811661885', last_modified=datetime.datetime(2025, 1, 15, 12, 18, 19), dataset_ids="['DS_mm4npn96qxo1_0']", chemical_formula_hill='B2Cl6Sb2', chemical_formula_reduced='BCl3Sb', chemical_formula_anonymous='A3BC', elements="['B', 'Cl', 'Sb']", elements_ratios='[0.2, 0.6, 0.2]', atomic_numbers='[5, 5, 51, 51, 17, 17, 17, 17, 17, 17]', nsites=10, nelements=3, nperiodic_dimensions=3, cell='[[11.181766970552692, 0.0, 0.0], [0.0, 4.847865509663038, 0.0], [0.0, 0.0, 5.141411249330318]]', dimension_types='[1, 1, 1]', pbc='[True, True, True]', names="['OMat24__agm002411254_ABC3_0_spg221_3_0_rattled-300_cgrzbm__file_ix_13106', 'OMat24__agm002411254_ABC3_0_spg221_3_0_rattled-300_cgrzbm__file_ix_48307']", labels='[]', metadata_id=None, metadata_path=None, metadata_size=None, structure_hash='100639255776800094606

In [37]:
# df4.write.mode('overwrite').parquet(f'combined_rows/combined_rows_incomplete')

In [18]:
df4.write.mode('overwrite').parquet("combined_rows/combined_rows_complete")

In [19]:
# dftest = spark.read.parquet(f'combined_rows/combined_rows_incomplete')
dftest = spark.read.parquet(f'combined_rows/combined_rows_complete')
dftest.count()

1182

In [39]:
dftest.first()

Row(id='CO_3016723021329385208855278', hash='3016723021329385208855278752111040259315261455261277072181072587164844007863398268613344050525989818600028256863362927911588833560093704230668798085095607', last_modified=datetime.datetime(2025, 2, 10, 15, 42, 37), dataset_ids="['DS_s6gf4z2hcjqy_0']", chemical_formula_hill='Al4Se2', chemical_formula_reduced='Al2Se', chemical_formula_anonymous='A2B', elements="['Al', 'Se']", elements_ratios='[0.6666666666666666, 0.3333333333333333]', atomic_numbers='[13, 13, 13, 13, 34, 34]', nsites=6, nelements=2, nperiodic_dimensions=3, cell='[[1.83940736, 5.65686674, -0.05486307], [-1.83940736, 5.65686674, -0.05486307], [0.0, 4.3111051, 5.92183747]]', dimension_types='[1, 1, 1]', pbc='[True, True, True]', names="['alexandria_3d__file_alex_go_aag_047__id_agm005420954__trajectory_0__frame_40', 'alexandria_3d__file_alex_go_aag_047__id_agm005420954__trajectory_1__frame_0', 'alexandria_3d__file_alex_go_aag_047__id_agm005420954__trajectory_2__frame_0']", labels=

In [20]:
arrow_schema = spark_schema_to_arrow_schema(StructType([StructField('$row_id', IntegerType(), True)]))

In [21]:
rows_to_remove = spark.read.parquet('/scratch/gw2338/vast/data-lake-main/spark/scripts/gw_scripts/fix_duplicate_co_ids/row_ids_to_drop.parquet')

In [22]:
drop_table = pa.table([np.array(df.select('$row_id').collect()).flatten().astype(np.uint64)], schema=arrow_schema)

In [24]:
len(drop_table)

2376

In [25]:
with sess.transaction() as tx:
    table = tx.bucket('colabfit').schema('dev').table('co_wip')
    results = table.delete(drop_table)

    
    # reader = table.select(predicate=table['id'].isin(ids), internal_row_id=True)
    # data = reader.read_all().to_struct_array().to_pandas()

In [28]:
cols =spark.table('ndb.colabfit.dev.co_wip').columns

In [33]:
arrow_schema = spark_schema_to_arrow_schema(config_schema)


In [35]:
arrow_rec_batch = pa.table(
                    [pa.array(col) for col in zip(*df4.collect())],
                    schema=arrow_schema,
                ).to_batches()

In [37]:
nrows = 0
with sess.transaction() as tx:
    table = tx.bucket('colabfit').schema('dev').table('co_wip')
    for batch in arrow_rec_batch:
        len_batch = batch.num_rows
        print(len_batch)
        table.insert(batch)
        nrows += len_batch

1182
